# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print dataset info
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id` fields.

Record sets structure the data in the Croissant schema. Let's print each record set and its fields/columns.

In [ ]:
# Overview of record sets and their fields, referring to all by @id
record_sets = metadata.record_sets
if not record_sets:
    print('No record sets declared in the schema metadata.')
else:
    for rs in record_sets:
        print(f"Record Set name: {rs.name} @id: {rs.id}")
        print(f"  Description: {rs.description}")
        print("  Fields (columns):")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
        print('-' * 60)

# For demonstration below, let's collect all record set @ids
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All access uses the record set and field `@id`s identified in the overview.

For demonstration, we'll load the first available record set.

In [ ]:
# Prepare all record sets as DataFrames
dataframes = {}
if record_set_ids:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set '{record_set_id}'. Columns: {df.columns.tolist()}")
    # For this notebook, we'll focus on the first available record set
    main_record_set = record_set_ids[0]
    print(f"\nSample records from '{main_record_set}':")
    display(dataframes[main_record_set].head())
else:
    print("No record sets are available. Please check dataset schema.")

## 4. Exploratory Data Analysis (EDA)
Explore and process the data using the DataFrame. Filtering, normalization, and grouping is performed using fields accessed via their exact `@id`s.

Below, select a numeric field and a group field (both by `@id`) for demonstration.

In [ ]:
if record_set_ids:
    # List all available column names by @id for the main record set
    columns = dataframes[main_record_set].columns.tolist()
    print(f"Columns in record set '{main_record_set}':\n{columns}")

    # Identify numeric fields by checking the schema (try common ones if unsure, e.g., peptide_count, coverage_pct, MW)
    # Here, try to automatically detect a numeric column for demo
    df = dataframes[main_record_set]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Selected numeric field for demo: {numeric_field}")
    else:
        print("No numeric fields detected.")
        numeric_field = None

    # Select a threshold
    if numeric_field:
        threshold = df[numeric_field].quantile(0.80)  # Top 20% threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where '{numeric_field}' > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try group by a non-numeric column, e.g. accession or description, by @id
        categorical_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        group_field = None
        if categorical_fields:
            group_field = categorical_fields[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by '{group_field}' (mean of '{numeric_field}'):")
            display(grouped_df.head())
        else:
            print("No categorical fields found for grouping.")
    else:
        print("Cannot perform EDA because no numeric field was found.")
else:
    print("No record sets loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll create a histogram for the selected numeric field (by `@id`) and a boxplot if grouping is performed.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field is not None:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot grouped by group_field if available
    if group_field is not None:
        plt.figure(figsize=(10,4))
        # If too many categories, only show top 10 by record count
        top_cats = df[group_field].value_counts().nlargest(10).index
        sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_cats)])
        plt.title(f"{numeric_field} by {group_field} (top 10 categories)")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
Summarized key findings and observations from the dataset exploration:

- The dataset contains mass spectrometry results structured in record sets. All data elements are referenced by their Croissant `@id`s for reproducibility.
- We loaded the records as DataFrames and identified candidate numeric fields suitable for quantitative analysis.
- Performed basic EDA: filtered values, normalized, and demonstrated grouping and aggregation by available metadata fields.
- Visualized numeric field distribution and per-category boxplots, which can reveal data quality and batch effects.
  
For deeper analysis, explore additional record sets, fields, or perform domain-specific data transformations as needed. All access should always use the unique `@id` references from the schema to ensure clarity and reproducibility.